# Phần 1: Phép khử Gauss và Các Ứng Dụng
Các hàm được cài đặt từ đầu (from scratch), không dùng NumPy/SciPy/SymPy cho thuật toán.

## Demo — Kiểm thử các trường hợp

In [10]:
import numpy as np
from gaussian import perform_gaussian_elimination
from determinant import calculate_matrix_determinant
from inverse import calculate_matrix_inverse
from rank_basis import calculate_rank_and_bases


def verify_solution(matrix_input_A, solution_result_x, constant_vector_b):
    print("\n" + "="*20 + " KIỂM CHỨNG VỚI NUMPY " + "="*20)
    matrix_A_numpy = np.array(matrix_input_A, dtype=float)
    vector_b_numpy = np.array(constant_vector_b, dtype=float)

    if isinstance(solution_result_x, list) and all(isinstance(val, (int, float)) for val in solution_result_x):
        residual_error = np.linalg.norm(matrix_A_numpy @ np.array(solution_result_x, dtype=float) - vector_b_numpy)
        print(f"1. Sai số hệ phương trình ||Ax - b|| = {residual_error:.2e}")
    elif isinstance(solution_result_x, tuple):
        solution_coefficients, free_column_indices = solution_result_x
        particular_solution_x0 = np.array([solution_coefficients[col_idx][0] for col_idx in range(len(solution_coefficients))])
        residual_error = np.linalg.norm(matrix_A_numpy @ particular_solution_x0 - vector_b_numpy)
        print(f"1. ||Ax₀ - b|| (nghiệm đặc biệt, ti=0) = {residual_error:.2e}")
    else:
        print(f"1. {solution_result_x}")

    if matrix_A_numpy.shape[0] == matrix_A_numpy.shape[1]:
        det_self_computed = calculate_matrix_determinant(matrix_input_A)
        det_numpy_val = np.linalg.det(matrix_A_numpy)
        print(f"2. Định thức: Tự tính = {det_self_computed:.4f}, NumPy = {det_numpy_val:.4f}")

    rank_self_computed, _, _, _ = calculate_rank_and_bases(matrix_input_A)
    rank_numpy_val = np.linalg.matrix_rank(matrix_A_numpy)
    print(f"3. Hạng ma trận: Tự tính = {rank_self_computed}, NumPy = {rank_numpy_val}")

    if matrix_A_numpy.shape[0] == matrix_A_numpy.shape[1] and abs(np.linalg.det(matrix_A_numpy)) > 1e-12:
        inv_result_self = calculate_matrix_inverse(matrix_input_A)
        if isinstance(inv_result_self, list):
            inv_error_val = np.linalg.norm(np.array(inv_result_self) - np.linalg.inv(matrix_A_numpy))
            print(f"4. Sai số ma trận nghịch đảo: {inv_error_val:.2e}")
        else:
            print(f"4. Nghịch đảo: {inv_result_self}")
    print("="*60)


In [11]:
print("="*55)
print("  Demo 1: Hệ 3x3 — nghiệm duy nhất")
print("="*55)
matrix_case1_A = [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]]
vector_case1_b = [8, -11, -3]
_, solution_case1_x, _ = perform_gaussian_elimination(matrix_case1_A, vector_case1_b)
print(f"  Nghiệm: x = {[round(val, 4) for val in solution_case1_x]}")
verify_solution(matrix_case1_A, solution_case1_x, vector_case1_b)


  Demo 1: Hệ 3x3 — nghiệm duy nhất
  Nghiệm: x = [2.0, 3.0, -1.0]

==================== KIỂM CHỨNG VỚI NUMPY ====================
1. Sai số hệ phương trình ||Ax - b|| = 8.88e-16
2. Định thức: Tự tính = -1.0000, NumPy = -1.0000
3. Hạng ma trận: Tự tính = 3, NumPy = 3
4. Sai số ma trận nghịch đảo: 4.93e-15


In [12]:
# ── Demo 2: Vô số nghiệm — 1 biến tự do ─────────────────────
print("="*55)
print("  Demo 2: Hệ 3x3 suy biến — vô số nghiệm")
print("="*55)
matrix_case2_A = [[1, 2, 3], [2, 4, 6], [1, 1, 1]]
vector_case2_b = [6, 12, 3]
_, solution_case2_x, _ = perform_gaussian_elimination(matrix_case2_A, vector_case2_b)
verify_solution(matrix_case2_A, solution_case2_x, vector_case2_b)


  Demo 2: Hệ 3x3 suy biến — vô số nghiệm

  ── Nghiệm tổng quát ──
  x1 = t1
  x2 = 3 - 2*t1
  x3 = t1

  ── Dạng vector ──
  x = (0, 3, 0)
      + t1 * (1, -2, 1)

==================== KIỂM CHỨNG VỚI NUMPY ====================
1. ||Ax₀ - b|| (nghiệm đặc biệt, ti=0) = 0.00e+00
2. Định thức: Tự tính = 0.0000, NumPy = 0.0000
3. Hạng ma trận: Tự tính = 2, NumPy = 2


In [14]:
# ── Demo 3: Vô số nghiệm — 2 biến tự do ─────────────────────
print("="*55)
print("  Demo 3: Hệ 3x3 suy biến nặng — 2 biến tự do")
print("="*55)
matrix_case3_A = [[1, 2, 4], [2, 4, 8], [3, 6, 12]]
vector_case3_b = [2, 4, 6]
_, solution_case3_x, _ = perform_gaussian_elimination(matrix_case3_A, vector_case3_b)


  Demo 3: Hệ 3x3 suy biến nặng — 2 biến tự do

  ── Nghiệm tổng quát ──
  x1 = 2 - 2*t1 - 4*t2
  x2 = t1
  x3 = t2

  ── Dạng vector ──
  x = (2, 0, 0)
      + t1 * (-2, 1, 0)
      + t2 * (-4, 0, 1)


In [15]:
print("="*55)
print("  Demo 4: Hệ vô nghiệm")
print("="*55)
matrix_case4_A = [[1, 2], [2, 4]]
vector_case4_b = [3, 7]
_, solution_case4_x, _ = perform_gaussian_elimination(matrix_case4_A, vector_case4_b)
print(f"  Kết quả: {solution_case4_x}")


  Demo 4: Hệ vô nghiệm
  Kết quả: Hệ phương trình vô nghiệm.


In [16]:
# ── Demo 5: Định thức ────────────────────────────────────────
print("="*55)
print("  Demo 5: Tính định thức")
print("="*55)
matrix_A3 = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]
print(f"  det(A) = {calculate_matrix_determinant(matrix_A3):.4f}  (NumPy: {np.linalg.det(matrix_A3):.4f})")

matrix_B = [
    [4, 3, 2, 1],
    [1, 2, 3, 4],
    [5, 0, 0, 5],
    [2, 2, 1, 1]
]
print(f"  det(B) = {calculate_matrix_determinant(matrix_B):.4f}  (NumPy: {np.linalg.det(matrix_B):.4f})")

matrix_singular_A = [[1, 2], [2, 4]]
print(f"  det(ma trận suy biến) = {calculate_matrix_determinant(matrix_singular_A)}  (đúng: 0)")


  Demo 5: Tính định thức
  det(A) = 1.0000  (NumPy: 1.0000)
  det(B) = 50.0000  (NumPy: 50.0000)
  det(ma trận suy biến) = 0.0  (đúng: 0)


In [17]:
# ── Demo 6: Ma trận nghịch đảo ───────────────────────────────
print("="*55)
print("  Demo 6: Ma trận nghịch đảo")
print("="*55)
matrix_inverse_A = [[1, 2, 3], [0, 1, 4], [5, 6, 0]]
inv_matrix_A = calculate_matrix_inverse(matrix_inverse_A)
if isinstance(inv_matrix_A, list):
    print("  A^-1 =")
    for row in inv_matrix_A:
        print(f"    {[round(val, 4) for val in row]}")
    error_val = np.linalg.norm(np.array(inv_matrix_A) - np.linalg.inv(matrix_inverse_A))
    print(f"  ||inv_self - inv_numpy|| = {error_val:.2e}")

print()
matrix_inverse_B = [
    [1, 0, 2, 1],
    [0, 1, 0, 2],
    [2, 1, 1, 0],
    [0, 1, 0, 1]
]

inv_matrix_B = calculate_matrix_inverse(matrix_inverse_B)

if isinstance(inv_matrix_B, list):
    print("  B^-1 =")
    for row in inv_matrix_B:
        print("    [" + " ".join(f"{val:>8.4f}" for val in row) + "]")
    
    error_val_B = np.linalg.norm(np.array(inv_matrix_B) - np.linalg.inv(matrix_inverse_B))
    print(f"\n  ||inv_self - inv_numpy|| = {error_val_B:.2e}")
else:
    print(f"  Kết quả: {inv_matrix_B}")

print()
print("  Ma trận không khả nghịch:")
matrix_singular_A2 = [[1, 2], [2, 4]]
print(f"  {calculate_matrix_inverse(matrix_singular_A2)}")


  Demo 6: Ma trận nghịch đảo
  A^-1 =
    [-24.0, 18.0, 5.0]
    [20.0, -15.0, -4.0]
    [-5.0, 4.0, 1.0]
  ||inv_self - inv_numpy|| = 1.77e-13

  B^-1 =
    [ -0.3333   1.0000   0.6667  -1.6667]
    [  0.0000  -1.0000   0.0000   2.0000]
    [  0.6667  -1.0000  -0.3333   1.3333]
    [ -0.0000   1.0000  -0.0000  -1.0000]

  ||inv_self - inv_numpy|| = 1.11e-16

  Ma trận không khả nghịch:
  Thông báo: Không có pivot tại cột 2. Ma trận không khả nghịch.


In [18]:
# ── Demo 7: Hạng và cơ sở ────────────────────────────────────
print("="*55)
print("  Demo 7: Hạng và cơ sở ")
print("="*55)
matrix_basis_A = [[1, 2, 3], [2, 4, 6], [1, 1, 1]]
rank_val, col_basis, row_basis, null_basis = calculate_rank_and_bases(matrix_basis_A)

print(f"Test case 1")
print(f"  Hạng: {rank_val}  (NumPy: {np.linalg.matrix_rank(matrix_basis_A)})")
print(f"  Cơ sở không gian cột  : {col_basis}")
print(f"  Cơ sở không gian dòng : {[[round(val, 4) for val in row] for row in row_basis]}")
print(f"  Cơ sở không gian nghiệm ({len(null_basis)} vector):")
for vector_v in null_basis:
    check_result = np.array(matrix_basis_A, dtype=float) @ np.array(vector_v)
    print(f"    {[round(val, 4) for val in vector_v]}  →  Av = {np.round(check_result, 10).tolist()}")

matrix_basis_rect = [
    [1, 2, 0, 1],
    [0, 1, 1, 0],
    [1, 3, 1, 1]
]
rank_val_r, col_basis_r, row_basis_r, null_basis_r = calculate_rank_and_bases(matrix_basis_rect)

print(f"\nTest case 2")
print(f"  Hạng: {rank_val_r}  (NumPy: {np.linalg.matrix_rank(matrix_basis_rect)})")
print(f"  Cơ sở không gian cột  : {col_basis_r}")
print(f"  Cơ sở không gian dòng : {[[round(val, 4) for val in row] for row in row_basis_r]}")
print(f"  Cơ sở không gian nghiệm ({len(null_basis_r)} vector):")
for vector_v in null_basis_r:
    check_result = np.array(matrix_basis_rect, dtype=float) @ np.array(vector_v)
    print(f"    {[round(val, 4) for val in vector_v]}  →  Av = {np.round(check_result, 10).tolist()}")


matrix_basis_rank1 = [
    [1, 1, 1],
    [2, 2, 2],
    [-1, -1, -1]
]
rank_val1, col_basis1, row_basis1, null_basis1 = calculate_rank_and_bases(matrix_basis_rank1)

print(f"\nTest case 3")
print(f"  Hạng: {rank_val1}  (NumPy: {np.linalg.matrix_rank(matrix_basis_rank1)})")
print(f"  Cơ sở không gian cột  : {col_basis1}")
print(f"  Cơ sở không gian dòng : {[[round(val, 4) for val in row] for row in row_basis1]}")
print(f"  Cơ sở không gian nghiệm ({len(null_basis1)} vector):")
for vector_v in null_basis1:
    check_result = np.array(matrix_basis_rank1, dtype=float) @ np.array(vector_v)
    print(f"    {[round(val, 4) for val in vector_v]}  →  Av = {np.round(check_result, 10).tolist()}")


  Demo 7: Hạng và cơ sở 
Test case 1
  Hạng: 2  (NumPy: 2)
  Cơ sở không gian cột  : [[1, 2, 1], [2, 4, 1]]
  Cơ sở không gian dòng : [[2, 4, 6], [0.0, -1.0, -2.0]]
  Cơ sở không gian nghiệm (1 vector):
    [1.0, -2.0, 1.0]  →  Av = [0.0, 0.0, 0.0]
Test case 2
  Hạng: 2  (NumPy: 2)
  Cơ sở không gian cột  : [[1, 0, 1], [2, 1, 3]]
  Cơ sở không gian dòng : [[1, 2, 0, 1], [0.0, 1.0, 1.0, 0.0]]
  Cơ sở không gian nghiệm (2 vector):
    [2.0, -1.0, 1.0, 0.0]  →  Av = [0.0, 0.0, 0.0]
    [-1.0, -0.0, 0.0, 1.0]  →  Av = [0.0, 0.0, 0.0]
Test case 3
  Hạng: 1  (NumPy: 1)
  Cơ sở không gian cột  : [[1, 2, -1]]
  Cơ sở không gian dòng : [[2, 2, 2]]
  Cơ sở không gian nghiệm (2 vector):
    [-1.0, 1.0, 0.0]  →  Av = [0.0, 0.0, 0.0]
    [-1.0, 0.0, 1.0]  →  Av = [0.0, 0.0, 0.0]
